In [2]:
!pip install -q langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 26.6 MB/s eta 0:00:00


In [3]:
import os
import json
from pathlib import Path
from getpass import getpass
from typing import Dict, TypedDict, Any, List, Literal

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Set model and notebook options

In [ ]:
PREVIEW_ROWS = 5
MAX_CATEGORICAL_VALUES = 12
MAX_CHARTS = 4
NARRATIVE_TABLE_LIMIT = 10

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 200)

### Set CSV input config

In [9]:
CSV_PATH = input("Enter Csv file path: ")

### Define structured schemas

In [12]:
class SchemaProfile(BaseModel):
    file_name: str
    row_count: str
    column_count: str
    numeric_columns: List[str]
    categorical_columns: List[str]
    datetime_columns: List[str]
    text_like_columns: List[str]
    parsed_datetime_columns: List[str]
    duplicate_rows: int
    missing_cells: int

class ChartRecommendation(BaseModel):
    chart_type: Literal["missing_bar", "histogram", "bar", "line", "scatter"]
    title: str
    columns: List[str]
    reason: str

class ChartPlan(BaseModel):
    charts: List[ChartRecommendation]

In [13]:
class NarrativeInsights(BaseModel):
    dataset_overview: str
    top_insights: List[str]
    data_quality_notes: List[str]
    recommended_focus_areas: List[str]
    
class BusinessSummary(BaseModel):
    executive_summary : str
    key_trends: List[str]
    risks: List[str]
    next_steps: List[str]

class FinalAnalysisPackage(BaseModel):
    schema : SchemaProfile
    recommended_charts: List[ChartRecommendation]
    narrative: NarrativeInsights
    business_summary: BusinessSummary

/tmp/ipykernel_2578/147651208.py:13: UserWarning: Field name "schema" in "FinalAnalysisPackage" shadows an attribute in parent "BaseModel"
  class FinalAnalysisPackage(BaseModel):


### Create helper functions

In [14]:
def smart_read_csv(path):
    return pd.read_csv(path, low_memory=False)

def try_parse_datetime_columns(df, threshold=0.8):
    df = df.copy()
    parsed_columns = []
    
    for col in df.columns:
        if df[col].dtype == "object":
            sample = df[col].dropna().astype(str).head(200)
            if len(sample) == 0:
                continue
            parsed_sample = pd.to_datetime(sample, errors="coerce")
            ratio = float(parsed_sample.notna().mean())
            if ratio >= threshold:
                full = pd.to_datetime(df[col], errors="coerce")
                if int(full.notna().sum()) > 0:
                    df[col] = full
                    parsed_columns.append(col)

    return df, parsed_columns

In [15]:
def infer_column_role(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    non_null = series.dropna()
    if len(non_null) == 0:
        return "categorical"
    avg_len = float(non_null.astype(str).str.len().mean())
    if avg_len > 30:
        return "text_like"
    return "categorical"

In [16]:
def build_schema_profile(df, file_name, parsed_datetime_columns):
    numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
    datetime_columns = df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()

    text_like_columns = []
    categorical_columns = []

    for col in df.columns:
        role = infer_column_role(df[col])
        if role == "text_like":
            text_like_columns.append(col)
        elif role == "categorical":
            categorical_columns.append(col)

    profile = SchemaProfile(
        file_name=file_name,
        row_count=int(df.shape[0]),
        column_count=int(df.shape[1]),
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        datetime_columns=datetime_columns,
        text_like_columns=text_like_columns,
        parsed_datetime_columns=parsed_datetime_columns,
        duplicate_rows=int(df.duplicated().sum()),
        missing_cells=int(df.isna().sum().sum())
    )
    return profile

In [19]:
def make_schema_df(df):
    rows = []
    for col in df.columns:
        rows.append(
            {
                "column": col,
                "dtype": str(df[col].dtype),
                "role": infer_column_role(df[col]),
                "non_null_count": int(df[col].notna().sum()),
                "missing_count": int(df[col].isna().sum()),
                "missing_pct": round(float(df[col].isna().mean() * 100), 2),
                "unique_count": int(df[col].nunique(dropna=True))
            }
        )
    return pd.DataFrame(rows).sort_values(["role", "column"]).reset_index(drop=True)

def make_missing_df(df):
    data = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_pct": (df.isna().mean() * 100).round(2).values
        }
    )
    return data.sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)

In [20]:
def make_numeric_summary_df(df):
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if not numeric_cols:
        return pd.DataFrame(
            columns=[
                "column", "count", "mean", "std", "min", "25%", "50%", "75%", "max",
                "missing_count", "missing_pct", "nunique", "skew"
            ]
        )

    summary = df[numeric_cols].describe().T.reset_index().rename(columns={"index": "column"})
    summary["missing_count"] = [int(df[col].isna().sum()) for col in summary["column"]]
    summary["missing_pct"] = [round(float(df[col].isna().mean() * 100), 2) for col in summary["column"]]
    summary["nunique"] = [int(df[col].nunique(dropna=True)) for col in summary["column"]]
    summary["skew"] = [round(float(df[col].skew()), 3) if pd.notna(df[col].skew()) else None for col in summary["column"]]
    return summary.sort_values(["std", "column"], ascending=[False, True]).reset_index(drop=True)

def make_categorical_summary_df(df):
    rows = []
    for col in df.columns:
        if infer_column_role(df[col]) == "categorical":
            value_counts = df[col].dropna().astype(str).value_counts()
            top_value = value_counts.index[0] if len(value_counts) > 0 else None
            top_frequency = int(value_counts.iloc[0]) if len(value_counts) > 0 else 0

            rows.append(
                {
                    "column": col,
                    "unique_count": int(df[col].nunique(dropna=True)),
                    "missing_count": int(df[col].isna().sum()),
                    "missing_pct": round(float(df[col].isna().mean() * 100), 2),
                    "top_value": top_value,
                    "top_frequency": top_frequency
                }
            )

    if not rows:
        return pd.DataFrame(columns=["column", "unique_count", "missing_count", "missing_pct", "top_value", "top_frequency"])

    return pd.DataFrame(rows).sort_values(["unique_count", "column"], ascending=[False, True]).reset_index(drop=True)

def make_datetime_summary_df(df):
    rows = []
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            non_null = df[col].dropna()
            rows.append(
                {
                    "column": col,
                    "missing_count": int(df[col].isna().sum()),
                    "missing_pct": round(float(df[col].isna().mean() * 100), 2),
                    "min_date": non_null.min() if len(non_null) > 0 else None,
                    "max_date": non_null.max() if len(non_null) > 0 else None,
                    "unique_count": int(df[col].nunique(dropna=True))
                }
            )

    if not rows:
        return pd.DataFrame(columns=["column", "missing_count", "missing_pct", "min_date", "max_date", "unique_count"])

    return pd.DataFrame(rows).sort_values(["column"]).reset_index(drop=True)

def make_correlations_df(df):
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if len(numeric_cols) < 2:
        return pd.DataFrame(columns=["x", "y", "correlation", "abs_correlation"])

    corr = df[numeric_cols].corr(numeric_only=True)
    rows = []

    for i, x in enumerate(numeric_cols):
        for y in numeric_cols[i + 1:]:
            value = corr.loc[x, y]
            if pd.notna(value):
                rows.append(
                    {
                        "x": x,
                        "y": y,
                        "correlation": round(float(value), 3),
                        "abs_correlation": round(abs(float(value)), 3)
                    }
                )

    if not rows:
        return pd.DataFrame(columns=["x", "y", "correlation", "abs_correlation"])

    return pd.DataFrame(rows).sort_values(["abs_correlation", "x", "y"], ascending=[False, True, True]).reset_index(drop=True)

def analyze_dataframe(df, file_name):
    working_df, parsed_datetime_columns = try_parse_datetime_columns(df)
    schema_model = build_schema_profile(working_df, file_name, parsed_datetime_columns)

    preview_df = working_df.head(PREVIEW_ROWS).copy()
    schema_df = make_schema_df(working_df)
    missing_df = make_missing_df(working_df)
    numeric_summary_df = make_numeric_summary_df(working_df)
    categorical_summary_df = make_categorical_summary_df(working_df)
    datetime_summary_df = make_datetime_summary_df(working_df)
    correlations_df = make_correlations_df(working_df)

    compact_payload = {
        "schema": schema_model.model_dump(),
        "missing_columns_top": missing_df.head(NARRATIVE_TABLE_LIMIT).to_dict(orient="records"),
        "numeric_summary_top": numeric_summary_df.head(NARRATIVE_TABLE_LIMIT).to_dict(orient="records"),
        "categorical_summary_top": categorical_summary_df.head(NARRATIVE_TABLE_LIMIT).to_dict(orient="records"),
        "datetime_summary": datetime_summary_df.head(NARRATIVE_TABLE_LIMIT).to_dict(orient="records"),
        "top_correlations": correlations_df.head(5).drop(columns=["abs_correlation"], errors="ignore").to_dict(orient="records")
    }

    return {
        "file_name": file_name,
        "df": working_df,
        "preview_df": preview_df,
        "schema_model": schema_model.model_dump(),
        "schema_df": schema_df,
        "missing_df": missing_df,
        "numeric_summary_df": numeric_summary_df,
        "categorical_summary_df": categorical_summary_df,
        "datetime_summary_df": datetime_summary_df,
        "correlations_df": correlations_df,
        "compact_payload": compact_payload
    }

def build_chart_plan(analysis):
    df = analysis["df"]
    missing_df = analysis["missing_df"]
    numeric_summary_df = analysis["numeric_summary_df"]
    categorical_summary_df = analysis["categorical_summary_df"]
    datetime_summary_df = analysis["datetime_summary_df"]
    correlations_df = analysis["correlations_df"]

    charts = []

    top_missing = missing_df[missing_df["missing_count"] > 0].head(10)
    if len(top_missing) > 0:
        charts.append(
            ChartRecommendation(
                chart_type="missing_bar",
                title="Missing Values by Column",
                columns=top_missing["column"].tolist(),
                reason="This quickly shows which columns need cleaning or imputation."
            )
        )

    if len(numeric_summary_df) > 0:
        num_col = numeric_summary_df.iloc[0]["column"]
        charts.append(
            ChartRecommendation(
                chart_type="histogram",
                title=f"Distribution of {num_col}",
                columns=[num_col],
                reason="This helps inspect spread, skewness, and unusual value concentration."
            )
        )

    filtered_cat = categorical_summary_df[
        (categorical_summary_df["unique_count"] >= 2) &
        (categorical_summary_df["unique_count"] <= MAX_CATEGORICAL_VALUES)
    ]
    if len(filtered_cat) > 0:
        cat_col = filtered_cat.sort_values(["unique_count", "top_frequency"], ascending=[False, False]).iloc[0]["column"]
        charts.append(
            ChartRecommendation(
                chart_type="bar",
                title=f"Top Categories in {cat_col}",
                columns=[cat_col],
                reason="This shows category concentration and distribution clearly."
            )
        )

    if len(datetime_summary_df) > 0 and len(numeric_summary_df) > 0:
        date_col = datetime_summary_df.iloc[0]["column"]
        num_col = numeric_summary_df.iloc[0]["column"]
        charts.append(
            ChartRecommendation(
                chart_type="line",
                title=f"{num_col} Over Time by {date_col}",
                columns=[date_col, num_col],
                reason="This helps reveal time trends and directional movement."
            )
        )

    strong_corr = correlations_df[correlations_df["abs_correlation"] >= 0.3]
    if len(strong_corr) > 0:
        row = strong_corr.iloc[0]
        charts.append(
            ChartRecommendation(
                chart_type="scatter",
                title=f"{row['x']} vs {row['y']}",
                columns=[row["x"], row["y"]],
                reason="This helps inspect relationship strength and possible correlation."
            )
        )

    unique_charts = []
    seen = set()

    for chart in charts:
        key = (chart.chart_type, tuple(chart.columns))
        if key not in seen:
            seen.add(key)
            unique_charts.append(chart)

    plan = ChartPlan(charts=unique_charts[:MAX_CHARTS])
    return plan.model_dump()

def plot_chart(df, chart):
    chart_type = chart["chart_type"]
    title = chart["title"]
    cols = chart["columns"]

    if chart_type == "missing_bar":
        subset = df[cols]
        counts = subset.isna().sum().sort_values(ascending=False)
        if int(counts.sum()) == 0:
            return
        plt.figure(figsize=(9, 4))
        counts.plot(kind="bar")
        plt.title(title)
        plt.xlabel("Column")
        plt.ylabel("Missing Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
        return

    if chart_type == "histogram":
        col = cols[0]
        series = df[col].dropna()
        if len(series) == 0:
            return
        plt.figure(figsize=(9, 4))
        plt.hist(series, bins=30)
        plt.title(title)
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()
        return

    if chart_type == "bar":
        col = cols[0]
        series = df[col].dropna().astype(str).value_counts().head(10)
        if len(series) == 0:
            return
        plt.figure(figsize=(9, 4))
        series.plot(kind="bar")
        plt.title(title)
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
        return

    if chart_type == "line":
        date_col, num_col = cols
        temp = df[[date_col, num_col]].dropna().sort_values(date_col)
        if len(temp) == 0:
            return
        temp = temp.groupby(date_col, as_index=False)[num_col].mean()
        plt.figure(figsize=(10, 4))
        plt.plot(temp[date_col], temp[num_col])
        plt.title(title)
        plt.xlabel(date_col)
        plt.ylabel(f"Mean {num_col}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
        return

    if chart_type == "scatter":
        x, y = cols
        temp = df[[x, y]].dropna()
        if len(temp) == 0:
            return
        plt.figure(figsize=(8, 5))
        plt.scatter(temp[x], temp[y], alpha=0.6)
        plt.title(title)
        plt.xlabel(x)
        plt.ylabel(y)
        plt.tight_layout()
        plt.show()
        return

def build_recommendation_df(chart_plan):
    charts = chart_plan.get("charts", [])
    if not charts:
        return pd.DataFrame(columns=["chart_type", "title", "columns", "reason"])
    rows = []
    for item in charts:
        rows.append(
            {
                "chart_type": item["chart_type"],
                "title": item["title"],
                "columns": ", ".join(item["columns"]),
                "reason": item["reason"]
            }
        )
    return pd.DataFrame(rows)

def build_markdown_report(final_output):
    lines = []
    lines.append("# Automatic Data Analysis Report")
    lines.append("")
    lines.append("## Dataset Overview")
    lines.append(final_output["narrative"]["dataset_overview"])
    lines.append("")
    lines.append("## Top Insights")
    for item in final_output["narrative"]["top_insights"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Data Quality Notes")
    for item in final_output["narrative"]["data_quality_notes"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Recommended Focus Areas")
    for item in final_output["narrative"]["recommended_focus_areas"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append(final_output["business_summary"]["executive_summary"])
    lines.append("")
    lines.append("## Key Trends")
    for item in final_output["business_summary"]["key_trends"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Risks")
    for item in final_output["business_summary"]["risks"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Next Steps")
    for item in final_output["business_summary"]["next_steps"]:
        lines.append(f"- {item}")
    lines.append("")
    lines.append("## Recommended Charts")
    for item in final_output["recommended_charts"]:
        lines.append(f"- **{item['title']}** ({item['chart_type']}) — {item['reason']}")
    return "\n".join(lines)

In [ ]:
class AnalysisState(TypedDict):
    raw_input: Dict[str, Any]
    analysis: Dict[str, Any]
    chart_plan: Dict[str, Any]
    narrative: Dict[str, Any]
    business_summary: Dict[str, Any]
    final_output: Dict[str, Any]
    report_markdown: str